In [ ]:
import sys
sys.path.insert(0, "../../../")

from src.agents import AgentDriver, AgentDriverConfig
from src.agents.utils import AgentConnectorConfig

import json
import re
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import PromptTemplate
from collections import defaultdict
from tqdm import tqdm

In [ ]:
BENCHMARK = "hotpotqa_distractor_validation" # TO CHANGE
FORMATTED_TRACES_PATH = f"../../../data/experiences_extraction/formatted_traces/{BENCHMARK}/without_queryprep/part1(casual_answer_generator).json"

In [ ]:
with open(FORMATTED_TRACES_PATH, 'r', encoding='utf-8') as fd:
    traces = json.loads(fd.read())

In [ ]:
agentd_config = AgentDriverConfig(
    agent_config=AgentConnectorConfig(
        credentials={ 'model': 'gemma3:12b','host': 'localhost','port': 11437 }
    )
)

In [ ]:
agent = AgentDriver.connect(agentd_config)

## Extracting Experiences

In [ ]:
RAW_EXPERIENCES_SAVE_PATH = f"../../../data/experiences_extraction/general_extraction(iteration#1)/{BENCHMARK}/raw_extracted_experiences.json"

In [ ]:
EXPERIENCE_EXTRACTION_SYSTEM_PROMPT = '''
You are an expert AI analyst reviewing successful step sequences from an AI agent execution.
Your task is to extract reusable, actionable step-level experiences that can guide future agent executions.
Focus on identifying specific patterns, techniques, and decision points that contributed to success.
ANALYSIS FRAMEWORK:
• STEP PATTERN ANALYSIS: Identify the specific sequence of actions that led to success
• DECISION POINTS: Highlight critical decisions made during these steps
• TECHNIQUE EFFECTIVENESS: Analyze why specific approaches worked well
• REUSABILITY: Extract patterns that can be applied to similar scenarios
EXTRACTION PRINCIPLES:
• Focus on TRANSFERABLE TECHNIQUES and decision frameworks
• Frame insights as actionable guidelines and best practices

# Original Query
{query}
# Step Sequence
{step_sequence}
# Generated Answer
{generated_answer}
# Outcome
This step sequence was part of a successful trajectory.

{output_format}
'''

In [ ]:
class ExperienceAnalysis(BaseModel):
    when_to_use: str = Field(
        description="Specific conditions: when this insight should be applied."
    )
    keywords: List[str] = Field(
        description="Relevant keywords from the 'when_to_use' field"
    )
    specific_experience: str = Field(
        description="Detailed description of the extracted successful pattern and why it works."
    )
    masked_experience: str = Field(
        description="Detailed description of the extracted successful pattern with masked named entities and specific terminology. For masking use '[Named Entity]' or '[General Term]' keywords."
    )
    generalized_experience: str = Field(
        description="Abstraction of the extracted specific successful pattern to create a more generalized and reusable experience representation."
    )
    tools_used: List[str] = Field(
        description="List of queries (with masked named entities and specialized terms) and/or tools/mechanics/operations that were used in a given step sequence. For masking use '[Named Entity]' or '[General Term]' keywords."
    )
    tags: List[str] = Field(
        description="Relevant tags about when to use extracted generalized experience."
    )


class Experiences(BaseModel):
    experiences: List[ExperienceAnalysis] = Field(
        description="Useful insights, extracted from the give step sequence."
    )

In [ ]:
experiencesextraction_parser = JsonOutputParser(pydantic_object=Experiences)
print(experiencesextraction_parser.get_format_instructions())

In [ ]:
experiencesextraction_template = PromptTemplate(
    template=EXPERIENCE_EXTRACTION_SYSTEM_PROMPT,
    input_variables=["query", "step_sequence", "generated_answer"], # Что должен предоставить пользователь
    partial_variables={         # Что заполняется автоматически
        "output_format": experiencesextraction_parser.get_format_instructions()
    }
)

In [ ]:
print(len(list(traces.keys())))

In [ ]:
extracted_experiences = dict()
max_trials = 3
parsing_errors_counter = 0
for i, query_id in enumerate(list(traces.keys())):
    print(f"{i}. query_id = {query_id}; query = '{traces[query_id]['question']}'")
    formatted_system_prompt = experiencesextraction_template.format(
        query=traces[query_id]['question'],
        step_sequence='\n'.join([f"{i+1}. {step_info['step_query']} --- {step_info['retrieved_info']}" for i, step_info in enumerate(traces[query_id]['search_steps'])]),
        generated_answer=traces[query_id]['generated_answer']
    )
    current_trial = 1
    is_output_parsed = False
    while current_trial <= max_trials:
        raw_response, stat = agent.generate(system_prompt=formatted_system_prompt, assistant_prompt="Extracted Experiences:\n")
        try:
            parsed_response = experiencesextraction_parser.invoke(raw_response)
        except OutputParserException as e:
            current_trial += 1
        else:
            is_output_parsed = True
            break

    if not is_output_parsed:
        print(f"!!! Parsing Error: query_id = {query_id} !!!")
        print(raw_response)
        extracted_experiences[query_id] = None
        parsing_errors_counter += 1
        continue
    else:
        print(json.dumps(parsed_response, ensure_ascii=False, indent=2))
        extracted_experiences[query_id] = parsed_response

    print("="*10)


In [ ]:
with open(RAW_EXPERIENCES_SAVE_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_experiences, ensure_ascii=False, indent=2))

In [ ]:
print(parsing_errors_counter)

## Validating Experiences

In [ ]:
EXPERIENCE_VALIDATION_SYSTEM_PROMPT = '''
You are an expert AI analyst tasked with validating the quality and usefulness of extracted step-level experiences.
Your task is to assess whether the extracted experience is actionable, accurate, and valuable for future agent executions.
VALIDATION CRITERIA:
* ACTIONABILITY: Is the experience specific enough to guide future actions?
* ACCURACY: Does the experience correctly reflect the patterns observed?
* RELEVANCE: Is the experience applicable to similar future scenarios?
* CLARITY: Is the experience clearly articulated and understandable?
* UNIQUENESS: Does the experience provide novel insights or common knowledge?

# Original Query
{query}
# Step Sequence
{step_sequence}
# Generated Answer
{generated_answer}

# Experience to Validate
Condition: 
{condition}
Experience Content: 
{experience_content}

{output_format}

Score should be between 0.0 (poor quality) and 1.0 (excellent quality).
Mark as invalid if score is below 0.3 or if there are fundamental issues with the experience.
'''

In [ ]:
class ExperienceValidation(BaseModel):
   is_valid: bool = Field(
      description="'True' if the given experience is meets all the validation criteria, else 'False'."
   )
   confidence: float = Field(
      description="Given experience quality.",
      ge=0.0,
      le=1.0
   )
   feedback: str = Field(
      description="Detailed explanation of validation decision."
   )
   recommendation: str = Field(
      description="Suggestions for improvement if applicable"
   )

In [ ]:
experiencevalidation_parser = JsonOutputParser(pydantic_object=ExperienceValidation)
print(experiencevalidation_parser.get_format_instructions())

In [ ]:
experiencevalidation_template = PromptTemplate(
    template=EXPERIENCE_VALIDATION_SYSTEM_PROMPT,
    input_variables=["query", "step_sequence", "generated_answer", "condition", "experience_content"], # Что должен предоставить пользователь
    partial_variables={         # Что заполняется автоматически
        "output_format": experiencevalidation_parser.get_format_instructions()
    }
)

In [ ]:
EXPERIENCE_TYPE = 'generalized_experience' # TO CHANGE 'specific_experience' | 'generalized_experience'
EXPERIENCES_VALIDATION_SAVE_PATH = f"../../../data/experiences_extraction/general_extraction(iteration#1)/{BENCHMARK}/{EXPERIENCE_TYPE}_validationinfo.json"

In [ ]:
experiences_validation_info = defaultdict(list)

In [ ]:
max_trials = 3
parsing_errors_counter = 0
no_experience_counter = 0
for i, query_id in enumerate(list(traces.keys())):
    for j, experience_info in enumerate(extracted_experiences[query_id]['experiences']):
        try:
            print(f"{i}.{j}. query_id = {query_id}; query = '{traces[query_id]['question']}'")
            print(f"'when_to_use': {experience_info['when_to_use']}")
            print(f'Experience: {experience_info[EXPERIENCE_TYPE]}')
        except KeyError:
            print(f"!!! No {EXPERIENCE_TYPE} !!!")
            no_experience_counter += 1
            experiences_validation_info[query_id].append(None)
            continue
        
        formatted_system_prompt = experiencevalidation_template.format(
            query=traces[query_id]['question'],
            step_sequence='\n'.join([f"{i+1}. {step_info['step_query']} --- {step_info['retrieved_info']}" for i, step_info in enumerate(traces[query_id]['search_steps'])]),
            generated_answer=traces[query_id]['generated_answer'],
            condition=experience_info['when_to_use'],
            experience_content=experience_info[EXPERIENCE_TYPE]
        )
        current_trial = 1
        is_output_parsed = False
        while current_trial <= max_trials:
            raw_response, stat = agent.generate(system_prompt=formatted_system_prompt, assistant_prompt="Experience Validation Result:\n")
            try:
                parsed_response = experiencevalidation_parser.invoke(raw_response)
            except OutputParserException as e:
                current_trial += 1
            else:
                is_output_parsed = True
                break

        if not is_output_parsed:
            print(f"!!! Parsing Error: query_id = {query_id} !!!")
            print(raw_response)
            experiences_validation_info[query_id].append(None)
            parsing_errors_counter += 1
            continue
        else:
            print(json.dumps(parsed_response, ensure_ascii=False, indent=2))
            experiences_validation_info[query_id].append(parsed_response)

        print("="*10)

In [ ]:
print(parsing_errors_counter)

In [ ]:
with open(EXPERIENCES_VALIDATION_SAVE_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(experiences_validation_info, ensure_ascii=False, indent=2))

In [ ]:
agent.close_connection()

In [ ]:
# техники использования метафоры для переиспользования предыдущего опыта с целью решения новых задач
# техники проведения аналогий с предыдущим опытом для решения новых задач
# техники проведения аналогий с предыдущим опытом для решения задач из другой предеметной области
# https://memgraph.com/blog/graph-visualization-in-python
# силлогизм
# метафора
# аналогия